In [ ]:
mport
torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as v2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from torch.nn import functional as F

embed_layer = nn.Embedding(10, 128)
y_emb = embed_layer(torch.tensor(4))
y_emb
y_emb.shape
z = torch.randn(1, 128)
z
inp = torch.cat([z, y_emb.unsqueeze(0)], dim=1)
inp
inp.shape


class Generator(nn.Module):
    def __init__(self, z_dim=128, out_channel=3, num_class=10, embed_dim=128):
        super(Generator, self).__init__()
        self.z_dim = z_dim
        self.num_class = num_class
        self.input_dim = self.z_dim + embed_dim

        self.embed = nn.Embedding(self.num_class, embed_dim)

        self.fc = nn.Sequential(
            nn.Linear(self.input_dim, 512),  # 512 > 4*4*32
            nn.ReLU(),
        )

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(16),

            nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(8),

            nn.ConvTranspose2d(in_channels=8, out_channels=out_channel, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, z, y):
        y_emb = self.embed(y)
        x = torch.cat([z, y_emb], dim=1)
        x = self.fc(x)
        x = x.view(x.shape[0], 32, 4, 4)
        x = self.deconv(x)

        return x


class Discriminator(nn.Module):
    def __init__(self, in_channels=3, num_class=10, embed_dim=128):
        super(Discriminator, self).__init__()

        self.embed = nn.Embedding(num_class, embed_dim)

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=4, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2),

        )  # > 4*4*32

        self.fake_classifier = nn.Sequential(
            nn.Linear(32 * 4 * 4, 1),
            nn.Sigmoid(),
        )

        self.fc_feat = nn.Sequential(
            nn.Linear(32 * 4 * 4, 128),
        )

        self.aux_classifier = nn.Sequential(
            nn.Linear(32 * 4 * 4, num_class),
            nn.Softmax(dim=1)
        )

    def forward(self, x, y):
        x = self.feature_extractor(x)
        x = x.view(x.shape[0], -1)  # 512

        rf_pred = self.fake_classifier(x)

        # feat_vec = self.fc_feat(x)
        # y_emb = self.embed(y)
        # feat_cat = torch.cat([feat_vec, y_emb], dim=1)
        cls_pred = self.aux_classifier(x)

        return rf_pred, cls_pred


def train():
    for real_imgs, real_y in tqdm(train_loader):
        real_imgs = real_imgs.to(device)
        real_y = real_y.to(device)

        batch_size = real_imgs.shape[0]

        # Train Discriminator
        optim_discriminator.zero_grad()
        noise = torch.randn(batch_size, noise_dim).to(device)
        fake_imgs = generator(noise, real_y).detach()

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        true_rf_pred, true_cls_pred = discriminator(real_imgs, real_y)
        fake_rf_pred, fake_cls_pred = discriminator(fake_imgs, real_y)

        loss_true_cls = F.cross_entropy(true_cls_pred, real_y)
        # loss_false_cls = F.cross_entropy(fake_cls_pred, real_y)
        loss_real = F.binary_cross_entropy(true_rf_pred, real_labels)
        loss_fake = F.binary_cross_entropy(fake_rf_pred, fake_labels)
        loss_discriminator = loss_real + loss_fake + loss_true_cls

        loss_discriminator.backward()
        optim_discriminator.step()

        # Train Generator
        optim_generator.zero_grad()
        noise = torch.randn(batch_size, noise_dim).to(device)
        fake_imgs = generator(noise, real_y)
        fake_preds, pred_cls_generated = discriminator(fake_imgs, real_y)

        loss_real_or_fake = F.binary_cross_entropy(fake_preds, real_labels)
        loss_classification = F.cross_entropy(pred_cls_generated, real_y)  # NEW EDITION
        loss_generator = loss_real_or_fake + loss_classification  # NEW EDITION
        loss_generator.backward()
        optim_generator.step()

    print(f"epoch: {epoch}, Loss D: {loss_discriminator.item()}, loss G: {loss_generator.item()}")


transform = v2.Compose([
    v2.RandomResizedCrop(32),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    v2.RandomRotation(15),
    v2.ToTensor(),
])
cifar10_train_dataset = torchvision.datasets.CIFAR10(root='./', download=True, train=True, transform=transform)
cifar10_test_dataset = torchvision.datasets.CIFAR10(root='./', download=True, train=False, transform=transform)

train_loader = DataLoader(cifar10_train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(cifar10_test_dataset, batch_size=64, shuffle=False)
noise_dim = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator = Generator(z_dim=128, out_channel=3, num_class=10, embed_dim=128).to(device)
discriminator = Discriminator(in_channels=3, num_class=10, embed_dim=128).to(device)

optim_generator = torch.optim.Adam(generator.parameters(), lr=1e-4)
optim_discriminator = torch.optim.Adam(discriminator.parameters(), lr=1e-4)

for epoch in range(100):
    train()
noise = torch.randn(1, noise_dim).to(device)
fake_imgs = generator(noise, torch.tensor([3]).to(device)).to(device)
plt.imshow(fake_imgs[0].permute(1, 2, 0).cpu().detach().squeeze())
from torchvision.models.vision_transformer import vit_b_16

vit_b_16
model = vit_b_16()
model
